# Diagnose the diffusivity gap before changing the model
Repository inspected: `rochan17/Constrictivity`, commit `78c06de6ac8bc3ae282bb21d4034de55559962c0`.

This notebook provides executable initial diagnostics and a protocol for the decisive local-pore test. It does **not** claim either hypothesis is established. Original microstructure arrays and cached fields were not present in the GitHub tree, so the real-data cells require your local files. The helper functions were checked on synthetic networks/fields; the repository-dependent reconstruction must be verified against your cached baseline.

1. Audit the saved scalar discrepancy, masks, convergence and normalization.
2. Compare **conservative voxel-face currents**, not cell-centred `J` or potential correlation alone.
3. Diagnose the centre-route hypothesis with a geometric screen and a specified local multi-port experiment.
4. Separate deleting an edge from reallocating its volume, with frozen-conductance controls.

Use the same axis, pore mask, applied reservoir potentials, and intrinsic diffusivity/conductivity. Here `D_rel = D_eff / D0 = 1/F`; all PNM calculations use voxel units and `sigma=1`.


In [ ]:
from pathlib import Path
import sys, json, pickle, copy
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import sparse
from scipy.sparse.linalg import spsolve
from scipy.sparse.csgraph import connected_components

REPO = Path.cwd()  # Set to your Constrictivity checkout.
# Explicit directories of real cached results (resolve version references first).
TAU_DIR = None    # e.g. REPO / 'data/Fine_microstructure_0/version4/taufactor'
CC_DIR = None     # e.g. REPO / 'data/Fine_microstructure_0/version4/berg_cc'
BODY_FILE = None  # e.g. REPO / 'data/Fine_microstructure_0/body_array.npy'
AXIS = 1         # 0=x, 1=y, 2=z; must match BOTH configs.
# Set these from the installed TauFactor solver's applied reservoir BCs.
# Do not use min/max of the interior field: those exclude boundary drops.
TAU_INLET = None
TAU_OUTLET = None


## 1. Scalar audit
Version 4 in the committed CSV contains ten Fine, ten Medium and four Coarse samples. Version numbers alone do not guarantee matched configurations across samples; examine config hashes and local config files. Repeated/cached versions are not independent observations.

The CSV has Fine PNM deficits of 38.5–45.2%, but Coarse PNM excesses of 26.2–104.9%. First verify that this is the result set you intend to diagnose.


In [ ]:
def scalar_audit(csv_path, version=4):
    d = pd.read_csv(csv_path)
    d = d.loc[d.version.eq(version)].copy()
    d['class'] = d.microstructure.str.split('_').str[0]
    d['D_PNM'] = 1 / d['berg_cc.F_total']
    for label, col in [('RN', 'berg_voxel.one_over_F'), ('PNM', 'D_PNM')]:
        d[label + '_error_pct'] = 100 * (d[col] / d['taufactor.D_eff'] - 1)
    return d, d.groupby('class')[['RN_error_pct','PNM_error_pct']].agg(['count','mean','min','max'])

if (REPO / 'all_runs_versions.csv').exists():
    scalar_rows, scalar_summary = scalar_audit(REPO / 'all_runs_versions.csv')
    display(scalar_summary)


## 2. Conservative flux helpers
For neighbouring pore voxels, current from voxel `a` toward voxel `b` is `D0*dx*(u[a]-u[b])` for cubic voxels. Use differences only across pore–pore faces. The helper also sums signed currents over **every face of each body-pair interface** and records interface energy.

A signed full-section current must be constant at steady state with insulating lateral boundaries; lateral flow cannot invalidate that conservation law. A changing sign across full sections means the field, axis, flux reconstruction or convergence needs attention.

`pipeline.run_taufactor` saves cell-centred averages and does not relabel vector components after `moveaxis`. This helper avoids both issues. For analysis normalize the field using the actual reservoir values to inlet=1, outlet=0.


In [ ]:
def face_diagnostics(u, pore, labels, axis):
    u = np.asarray(u, dtype=float)
    pore = np.asarray(pore, dtype=bool)
    labels = np.asarray(labels)
    assert u.shape == pore.shape == labels.shape and u.ndim == 3
    assert np.isfinite(u[pore]).all()
    div = np.zeros_like(u)
    pieces = []
    section = None
    for ax in range(3):
        a = [slice(None)]*3; b = [slice(None)]*3
        a[ax] = slice(None,-1); b[ax] = slice(1,None)
        a,b = tuple(a),tuple(b)
        valid = pore[a] & pore[b]
        q = np.zeros_like(u[a])
        q[valid] = u[a][valid] - u[b][valid]  # D0=dx=1
        div[a] += q; div[b] -= q
        if ax == axis:
            section = q.sum(axis=tuple(k for k in range(3) if k != ax))
        la,lb = labels[a],labels[b]
        cut = valid & (la>0) & (lb>0) & (la!=lb)
        x,y = la[cut].astype(int),lb[cut].astype(int)
        oriented = np.where(x<y, q[cut], -q[cut])
        if len(x):
            pieces.append(pd.DataFrame({'body_lo':np.minimum(x,y),
                'body_hi':np.maximum(x,y), 'q_ref':oriented,
                'abs_face_q':np.abs(oriented), 'interface_energy':q[cut]**2,
                'n_faces':np.ones(len(x),dtype=int)}))
    cols=['body_lo','body_hi','q_ref','abs_face_q','interface_energy','n_faces']
    pairs = (pd.concat(pieces).groupby(['body_lo','body_hi'],as_index=False).sum()
             if pieces else pd.DataFrame(columns=cols))
    inside = pore.copy()
    low=[slice(None)]*3; high=[slice(None)]*3
    low[axis]=0; high[axis]=-1
    inside[tuple(low)]=False; inside[tuple(high)]=False
    return pairs, section, div[inside]


def solve_graph(conns, g, n, inlet, outlet):
    """Dirichlet solve, inlet=1/outlet=0; gauge floating components only.
    Attached dead ends remain in the solve at their physical potential.
    """
    conns=np.asarray(conns,int); g=np.asarray(g,float)
    inlet=np.asarray(inlet,int); outlet=np.asarray(outlet,int)
    assert len(inlet) and len(outlet) and not np.intersect1d(inlet,outlet).size
    assert np.isfinite(g).all() and (g>=0).all()
    assert (conns>=0).all() and (conns<n).all()
    i,j=conns.T
    L=sparse.coo_matrix((np.r_[g,g,-g,-g],
        (np.r_[i,j,i,j],np.r_[i,j,j,i])),shape=(n,n)).tocsr()
    pos=g>0
    adj=sparse.coo_matrix((np.ones(2*pos.sum()),
        (np.r_[i[pos],j[pos]],np.r_[j[pos],i[pos]])),shape=(n,n)).tocsr()
    _,component=connected_components(adj,directed=False)
    anchored=np.unique(component[np.r_[inlet,outlet]])
    floating=np.flatnonzero(~np.isin(component,anchored))
    fixed=np.unique(np.r_[inlet,outlet,floating])
    free=np.setdiff1d(np.arange(n),fixed)
    u=np.zeros(n); u[inlet]=1
    if len(free): u[free]=spsolve(L[free][:,free], -L[free][:,fixed]@u[fixed])
    q=g*(u[i]-u[j]); divergence=L@u
    total=float(divergence[inlet].sum())
    energy=float(np.sum(g*(u[i]-u[j])**2))
    assert np.isfinite(u).all()
    assert np.isclose(total,energy,rtol=1e-7,atol=1e-12)
    return {'u':u,'q':q,'G_eff':total,'energy':energy,
            'outlet_current':-float(divergence[outlet].sum())}


## 3. Load fields and reproduce the actual PNM baseline
Load only trusted local pickle files. Set the three file/directory inputs above to enable this cell.

The current pipeline merges boundary-only bodies and short-circuit bodies before extraction. The existing `prune_and_recompute` baseline omits those merges. This cell reproduces the current pipeline's merge steps, retains the processed labels for interface matching, and checks cached throat IDs and conductances. A failed check is a baseline mismatch to investigate, not a check to remove.

Keep `voxel_size=1` here. In the inspected `berg_cc` code, `voxel_size` scales the sample length but not all network geometry/volume arrays, so non-unit values need a separate units audit.


In [ ]:
ready = all(x is not None for x in (TAU_DIR, CC_DIR, BODY_FILE, TAU_INLET, TAU_OUTLET))
if ready:
    sys.path.insert(0,str(REPO))
    import particulate_claude as pc
    import berg_cc
    TAU_DIR,CC_DIR=Path(TAU_DIR),Path(CC_DIR)
    with open(TAU_DIR/'full_result.pkl','rb') as f: tf=pickle.load(f)
    with open(CC_DIR/'full_result.pkl','rb') as f: cc=pickle.load(f)
    cfg=json.loads((CC_DIR/'config.json').read_text())
    tc=json.loads((TAU_DIR/'config.json').read_text())
    tm=json.loads((TAU_DIR/'metrics.json').read_text())
    cm=json.loads((CC_DIR/'metrics.json').read_text())
    assert tc.get('axis',0)==AXIS==pc.DIRECTION_TO_AXIS[cfg.get('direction','x')]
    assert cfg.get('voxel_size',1)==1, 'Use consistent voxel units for this diagnostic.'
    raw=np.load(BODY_FILE)
    labels,_,_=pc.merge_boundary_only_bodies_iterative(raw.copy(),axis=AXIS,verbose=False)
    labels,_=pc.merge_short_circuit_bodies(labels,axis=AXIS,verbose=False)
    pore=np.asarray(tf['pore_mask'],bool)
    assert np.array_equal(pore,labels>0), 'TauFactor and body masks differ.'
    assert TAU_INLET != TAU_OUTLET
    uref=(np.asarray(tf['potential_field'],float)-TAU_OUTLET)/(TAU_INLET-TAU_OUTLET)
    net=pc.extract_throats_from_bodies_voxel_parallel(labels,
        min_throat_size=cfg.get('min_throat_size',5),
        dilate_both=cfg.get('dilate_both',True),n_jobs=1,
        direction=cfg.get('direction','x'),surface_axis=cfg.get('surface_axis'),
        skip_surface_surface=cfg.get('skip_surface_surface',True),verbose=False)
    net,_=pc.calculate_network_effective_properties(net,
        area_method=cfg.get('area_method','vox_projection'),alpha=cfg.get('alpha',.5),
        split_volume_equal=cfg.get('split_volume_equal',True),verbose=False)
    a=berg_cc.build_network_arrays(net,area_method=cfg.get('area_method','vox_projection'))
    g,res=berg_cc.calculate_conductances(a,sigma=1,verbose=False)
    oldg,_=berg_cc.calculate_conductances(cc['arrays'],sigma=1,verbose=False)
    assert np.array_equal(a['body_ids'],cc['arrays']['body_ids'])
    assert np.array_equal(a['throat_ids'],cc['arrays']['throat_ids'])
    assert np.array_equal(a['conns'],cc['arrays']['conns'])
    assert np.allclose(g,oldg,rtol=1e-8,atol=1e-12), 'Rebuilt geometry differs from cache.'
    inlet,outlet=berg_cc.get_inlet_outlet_indices(a,labels,direction=cfg.get('direction','x'))
    base=solve_graph(a['conns'],g,len(a['body_ids']),inlet,outlet)
    L=labels.shape[AXIS]; area=labels.size/L
    D_pnm=base['G_eff']*L/area
    cached_direct=cm['effective_conductance']/cfg.get('sigma',1)*L/area
    display(pd.Series({'D_TauFactor':tm['D_eff'],'D_PNM_direct':D_pnm,
        'D_PNM_cached_direct':cached_direct,'D_PNM_from_F':1/cm['F_total']}))
    assert np.isclose(D_pnm,cached_direct,rtol=1e-6)
else:
    print('Set paths and actual TauFactor reservoir values to run real-data cells.')


## 4. Where do currents disagree?
Compare absolute currents at equal intrinsic conductivity and unit reservoir drop. Separately compare current fractions `q / total_current` to distinguish scale error from routing error. Do not compare only normalized fractions: they hide a uniform conductance deficit.

Unmatched interfaces are evidence about extraction: the PNM may contain connections with no direct voxel-face interface or omit real interfaces. Never silently treat these as zero-error matches. Dilation-based throat extraction can create indirect connections that merit inspection.


In [ ]:
if ready:
    pairs,sections,voxel_residual=face_diagnostics(uref,pore,labels,AXIS)
    qmean=sections.mean()
    display(pd.Series({'section_current_mean':qmean,
        'section_range_over_mean':np.ptp(sections)/max(abs(qmean),1e-30),
        'max_voxel_residual_over_total':np.max(np.abs(voxel_residual),initial=0)/max(abs(qmean),1e-30),
        'D_from_internal_faces':qmean*L/area,'D_TauFactor':tm['D_eff']}))
    plt.plot(sections); plt.xlabel('Internal section'); plt.ylabel('Signed total current'); plt.show()
    ids=a['body_ids']; b1,b2=ids[a['conns']].T
    edges=pd.DataFrame({'throat_id':a['throat_ids'], 'body_lo':np.minimum(b1,b2),
        'body_hi':np.maximum(b1,b2),'q_pnm':np.where(b1<b2,base['q'],-base['q']),
        'g':g,'power':base['q']**2/g,'fallback':a['is_fallback'],
        'volume':a['V_body1']+a['V_body2']+a['throat_volume']})
    edges['power_share']=edges.power/edges.power.sum()
    matched=edges.merge(pairs,on=['body_lo','body_hi'],how='outer',indicator=True)
    display(matched['_merge'].value_counts())
    view=matched.query('_merge == "both"').copy()
    q=view.q_ref.to_numpy(); qp=view.q_pnm.to_numpy()
    scale=float(q@qp/(q@q)) if q@q else np.nan
    residual=np.linalg.norm(qp-scale*q)/max(np.linalg.norm(qp),1e-30)
    print('PNM/ref fitted current scale:',scale,' residual after scale fit:',residual)
    fig,axs=plt.subplots(1,2,figsize=(10,4))
    axs[0].scatter(q,qp,s=12); axs[0].set(xlabel='Voxel interface current',ylabel='PNM current')
    axs[1].scatter(q/qmean,qp/base['G_eff'],s=12)
    axs[1].set(xlabel='Voxel current / total',ylabel='PNM current / total')
    plt.tight_layout();plt.show()


## 5. Hypothesis 1: centre-route over-resistance
First screen: for incoming port `a`, outgoing port `b`, and body centre `c`, compute
`detour = (|a-c| + |b-c|) / |a-b|`.
Use actual throat centres; straight chords through solid are not admissible paths. This ratio is only a **candidate score**, not measured tortuosity. Product weighting of incoming/outgoing currents below assumes complete mixing and is only a screening convention. Large detour in low-current regions is not strong evidence.

Decisive test (not implemented by the screen): choose interior high-current bodies, retain their actual pore voxels and throat ports, and solve independent local diffusion load cases. With `m` ports, impose one port at 1 and the rest at 0 for `m-1` independent cases; solid walls are insulating. Use the **same port boundaries** for the voxel and PNM comparison, including consistently matched half-face resistances. This identifies the port response `q=K*u`. A single-centre star implies `K = diag(h) - h*h.T/sum(h)`.

Fit positive arm conductances `h` to the voxel response. If the fitted star works but the geometric PNM star does not, conductance parameterization is implicated. If even a fitted star fails across independent load cases, the single-centre representation is inadequate. The test has greatest discriminatory power for 4+ ports. A single imposed whole-sample field is not enough to validate a general port response. Compare body resistance, port geometry, and throughput; degree alone does not establish a detour.


In [ ]:
def detour_screen(network, arrays, currents):
    rows=[]
    for idx,bid in enumerate(arrays['body_ids']):
        body=network.bodies[int(bid)]
        if body.is_surface: continue
        incident=np.flatnonzero((arrays['conns']==idx).any(axis=1))
        inc=[];out=[]
        for e in incident:
            signed=currents[e] if arrays['conns'][e,0]==idx else -currents[e]
            centre=np.asarray(network.throats[int(arrays['throat_ids'][e])].center,float)
            (out if signed>0 else inc).append((centre,abs(signed)))
        weighted=0.; weight=0.; maxratio=np.nan
        c=np.asarray(body.centroid,float)
        ratios=[]
        for p,qi in inc:
            for r,qo in out:
                chord=np.linalg.norm(p-r)
                if chord<=0 or qi*qo==0:continue
                ratio=(np.linalg.norm(p-c)+np.linalg.norm(r-c))/chord
                w=qi*qo;weighted+=w*ratio;weight+=w;ratios.append(ratio)
        rows.append({'body_id':int(bid),'degree':len(incident),
            'throughput':sum(q for _,q in inc),
            'detour_proxy':weighted/weight if weight else np.nan,
            'max_detour_proxy':max(ratios,default=np.nan)})
    return pd.DataFrame(rows)

if ready:
    detours=detour_screen(net,a,base['q'])
    display(detours.sort_values('throughput',ascending=False).head(15))
    # Sensitivity to body-arm resistance only: an intervention, not proof of detours.
    sweep=[]
    for eta in [1.,.75,.5,.25,.1,.01]:
        trialg=1/(eta*(res['R1']+res['R2'])+res['R_throat'])
        result=solve_graph(a['conns'],trialg,len(a['body_ids']),inlet,outlet)
        sweep.append({'body_R_multiplier':eta,'D_rel':result['G_eff']*L/area})
    display(pd.DataFrame(sweep))
    print('Fraction of baseline dissipation in body arms:',
        np.sum(base['q']**2*(res['R1']+res['R2']))/base['energy'])


The resistance sweep asks whether body-arm resistance has enough influence to explain the gap. It does **not** isolate geometric detour from a small assigned area. Also, the model's throat segment length depends on centre distances, so this is only a body-arm diagnostic, not an upper bound for every possible centre-route effect. Shortening plotted streamtubes after solving cannot change direct `D_rel`.

## 6. Hypothesis 2: assigning volume to weakly useful connections
Steady-state pore volume affects transport here through the assigned areas/lengths, not as storage capacity. Equal splitting gives `V_body / degree`; the fallback has `g = sigma * V_total / L_total**2`. Thus degree can affect conductance through volume splitting as well as through centre geometry.

Use dissipation `P_e=q_e**2/g_e`, not simply low current, to screen candidate connections. At fixed boundaries, the exact linear-network sensitivity is `d log(G_eff) / d log(g_e) = P_e / sum(P)`. However, low dissipation does not prove a connection's volume is physically available to another connection. For that, inspect voxel flux or trace flux tubes; retaining stagnant volume can be physically correct.

Run three controls for the **same preselected edge set**:
- baseline;
- remove edges while freezing all survivor conductances;
- remove the same edges, recompute volume shares and effective geometry, and solve again.

Fixed-conductance removal cannot increase effective conductance. An increase only after recomputation demonstrates an allocation/geometry effect in this model. It does not by itself prove the reassigned volume represents the voxel solution correctly. Keep random controls and compare to independent voxel currents; do not repeatedly choose the set that best matches the desired scalar.


In [ ]:
def pruning_controls(network, arrays, g, inlet, outlet, remove_ids, cfg):
    import particulate_claude as pc
    import berg_cc
    removed=np.isin(arrays['throat_ids'],list(remove_ids))
    n=len(arrays['body_ids'])
    baseline=solve_graph(arrays['conns'],g,n,inlet,outlet)
    frozen=solve_graph(arrays['conns'],np.where(removed,0.,g),n,inlet,outlet)
    assert frozen['G_eff'] <= baseline['G_eff']*(1+1e-7)+1e-12
    pruned=copy.deepcopy(network)
    removed_set=set(int(x) for x in arrays['throat_ids'][removed])
    for tid in removed_set:
        throat=pruned.throats[tid]
        for bid,other in ((throat.body1_id,throat.body2_id),(throat.body2_id,throat.body1_id)):
            pruned.bodies[bid].connected_bodies.remove(other)
        del pruned.throats[tid]
    for body in pruned.bodies.values():
        body.connected_throats=[tid for tid in body.connected_throats if tid not in removed_set]
        body.connections_to_throats.clear()
    pruned,_=pc.calculate_network_effective_properties(pruned,
        area_method=cfg.get('area_method','vox_projection'),alpha=cfg.get('alpha',.5),
        split_volume_equal=cfg.get('split_volume_equal',True),verbose=False)
    ar=berg_cc.build_network_arrays(pruned,area_method=cfg.get('area_method','vox_projection'))
    gr,_=berg_cc.calculate_conductances(ar,sigma=1,verbose=False)
    assert np.array_equal(ar['body_ids'],arrays['body_ids'])
    reallocated=solve_graph(ar['conns'],gr,n,inlet,outlet)
    return pd.DataFrame([{'case':name,'G_eff':sol['G_eff']} for name,sol in
        [('baseline',baseline),('removed_frozen_g',frozen),('removed_reallocated',reallocated)]]), ar

if ready:
    # A predeclared screen: bottom 5% by power share; also run a matched random set.
    nremove=max(1,int(.05*len(edges)))
    candidate_ids=edges.nsmallest(nremove,'power_share').throat_id.to_numpy()
    random_ids=np.random.default_rng(17).choice(a['throat_ids'],nremove,replace=False)
    selected=edges.throat_id.isin(candidate_ids)
    print('Selected volume fraction:',edges.loc[selected,'volume'].sum()/edges.volume.sum())
    print('Selected dissipation fraction:',edges.loc[selected,'power'].sum()/edges.power.sum())
    control,pruned_arrays=pruning_controls(net,a,g,inlet,outlet,candidate_ids,cfg)
    random_control,_=pruning_controls(net,a,g,inlet,outlet,random_ids,cfg)
    control['D_rel']=control.G_eff*L/area
    random_control['D_rel']=random_control.G_eff*L/area
    display(control,random_control)
    print('Fallback fraction before/after:',a['is_fallback'].mean(),pruned_arrays['is_fallback'].mean())


## 7. Stronger volume-allocation test (protocol)
For an interior body, treat its incident allocated volumes `V_ik` as independent inputs with `sum_k V_ik=V_i`. Transfer a small amount from candidate connection `k` to connection `l`, rebuild only affected conductances with the **existing** three-segment/fallback rules, and re-solve. Keep labels, connections, centres, throat areas and boundary conditions fixed. Sweep both signs and decreasing perturbation sizes; record fallback switches.

The first-order global change is
`delta G_eff = sum_e ((u_i-u_j)/DeltaU)**2 * delta g_e`.
Use finite differences of the existing geometry formula to get `delta g_e`; do not assume its dependence is simply linear outside the fallback regime. A reproducible positive sensitivity says the current allocation depresses conductance locally. Independent voxel evidence must still show that the donor was overallocated; otherwise you have only optimized the network.

Then use a 2x2 experiment (original/locally validated pore response x original/voxel-supported volume allocation) to measure each contribution and interaction. Compare absolute interface-current error as well as global `D_rel`.

## Interpretation checklist
- RN vs TauFactor residual: tighten convergence on the coarse examples before using either as reference; compare inlet/outlet totals and section currents.
- Uniform current scale error with good routing: conductance scale/area/length assignment is a strong lead; matching potentials cannot rule it out.
- Large geometric detour + local voxel response inconsistent with the centre-star: evidence for hypothesis 1.
- Big improvement only in reallocated pruning, plus independent voxel evidence of excess donor volume: evidence for hypothesis 2.
- Coarse PNM already too conductive: inspect missing/spurious connections, boundary-body merging and pinning, surface-surface exclusions, throat areas and fallback fractions before invoking a mechanism that only increases conductance.
- Keep `D_rel` from boundary current separate from `tau_sq_c`, `C_c`, and streamtube-volume bookkeeping. In this repository Berg's `tau_sq_c` uses the inverse squared length convention; it is not TauFactor's tortuosity factor.

### Source locations
- [Pipeline and field export](https://github.com/rochan17/Constrictivity/blob/78c06de6ac8bc3ae282bb21d4034de55559962c0/pipeline.py)
- [Geometry, volume shares and fallback](https://github.com/rochan17/Constrictivity/blob/78c06de6ac8bc3ae282bb21d4034de55559962c0/particulate_claude.py)
- [Conductances, boundaries and Berg reporting](https://github.com/rochan17/Constrictivity/blob/78c06de6ac8bc3ae282bb21d4034de55559962c0/berg_cc.py)
- [Current comparison implementation](https://github.com/rochan17/Constrictivity/blob/78c06de6ac8bc3ae282bb21d4034de55559962c0/analysis/throat_analysis.py)
- [Existing pruning experiment](https://github.com/rochan17/Constrictivity/blob/78c06de6ac8bc3ae282bb21d4034de55559962c0/analysis/artificial/throat_removal.py)
